# 02 — Data Cleaning & Panel Construction

This notebook takes the raw data collected in `01_data_collection.ipynb` and:
1. Cleans and standardizes each dataset
2. Merges everything into a state × year panel (2010–2022)
3. Constructs treatment variables for DiD analysis
4. Validates the final panel

**Input:** Files in `data/raw/`  
**Output:** `data/processed/analysis_panel.csv`

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Make src/ importable from the notebooks/ directory
sys.path.insert(0, '..')
from src.data_utils import parse_cdc_wonder_csv, parse_natality_csv, fetch_acs_data

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Study parameters
YEAR_START = 2010
YEAR_END = 2022
YEARS = list(range(YEAR_START, YEAR_END + 1))

print(f"Panel period: {YEAR_START}–{YEAR_END} ({len(YEARS)} years)")

Panel period: 2010–2022 (13 years)


---
## 1. Load & Clean Medicaid Expansion Status

In [2]:
df_expansion = pd.read_csv(f'{RAW_DIR}/kff_expansion_status.csv')

print(f"Shape: {df_expansion.shape}")
print(f"\nExpansion breakdown:")
print(df_expansion['cohort'].value_counts())

# Ensure state_fips is zero-padded string
df_expansion['state_fips'] = df_expansion['state_fips'].astype(str).str.zfill(2)

df_expansion.head()

Shape: (51, 6)

Expansion breakdown:
cohort
Early (2014)    27
Never           10
Late (2015)      3
Late (2020)      3
Late (2016)      2
Late (2019)      2
Late (2021)      2
Late (2023)      2
Name: count, dtype: int64


,state,state_fips,expansion_year,expansion_date,ever_expanded,cohort
0,Alabama,01,0,NaN,0,Never
1,Alaska,02,2015,2015-09-01,1,Late (2015)
2,Arizona,04,2014,2014-01-01,1,Early (2014)
3,Arkansas,05,2014,2014-01-01,1,Early (2014)
4,California,06,2014,2014-01-01,1,Early (2014)


---
## 2. Create the Base Panel Skeleton

Create a balanced panel with every state × year combination, then merge treatment variables.

In [3]:
# Create state × year skeleton
states = df_expansion[['state', 'state_fips']].copy()
years_df = pd.DataFrame({'year': YEARS})

# Cross join
panel = states.merge(years_df, how='cross')
print(f"Panel skeleton: {panel.shape[0]} rows ({len(states)} states × {len(YEARS)} years)")

# Merge expansion status
panel = panel.merge(
    df_expansion[['state_fips', 'expansion_year', 'ever_expanded', 'cohort']],
    on='state_fips',
    how='left'
)

# Construct treatment timing variables
panel['post_expansion'] = np.where(
    (panel['ever_expanded'] == 1) & (panel['year'] >= panel['expansion_year']),
    1, 0
)

# Years relative to expansion (event time)
# For never-treated states, this stays NaN
panel['event_time'] = np.where(
    panel['ever_expanded'] == 1,
    panel['year'] - panel['expansion_year'],
    np.nan
)

# Cohort group for Callaway-Sant'Anna (0 = never treated)
panel['cohort_group'] = np.where(
    panel['ever_expanded'] == 1,
    panel['expansion_year'],
    0
)

# DiD interaction term (for simple 2x2 DiD)
panel['treat_x_post'] = panel['ever_expanded'] * panel['post_expansion']

print(f"\nTreatment variable check:")
print(f"  post_expansion = 1: {panel['post_expansion'].sum()} obs")
print(f"  post_expansion = 0: {(panel['post_expansion'] == 0).sum()} obs")
print(f"  Ever expanded: {panel['ever_expanded'].sum()} obs")
print(f"  Never expanded: {(panel['ever_expanded'] == 0).sum()} obs")

panel.head(10)

Panel skeleton: 663 rows (51 states × 13 years)

Treatment variable check:
  post_expansion = 1: 302 obs
  post_expansion = 0: 361 obs
  Ever expanded: 533 obs
  Never expanded: 130 obs


,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post
0,Alabama,01,2010,0,0,Never,0,NaN,0,0
1,Alabama,01,2011,0,0,Never,0,NaN,0,0
2,Alabama,01,2012,0,0,Never,0,NaN,0,0
3,Alabama,01,2013,0,0,Never,0,NaN,0,0
4,Alabama,01,2014,0,0,Never,0,NaN,0,0
5,Alabama,01,2015,0,0,Never,0,NaN,0,0
6,Alabama,01,2016,0,0,Never,0,NaN,0,0
7,Alabama,01,2017,0,0,Never,0,NaN,0,0
8,Alabama,01,2018,0,0,Never,0,NaN,0,0
9,Alabama,01,2019,0,0,Never,0,NaN,0,0


In [4]:
# Validate: check a known state
print("Louisiana (expanded July 2016):")
la = panel[panel['state'] == 'Louisiana'][['state', 'year', 'expansion_year', 'post_expansion', 'event_time']]
display(la)

print("\nTexas (never expanded):")
tx = panel[panel['state'] == 'Texas'][['state', 'year', 'expansion_year', 'post_expansion', 'event_time']]
display(tx)

Louisiana (expanded July 2016):


,state,year,expansion_year,post_expansion,event_time
234,Louisiana,2010,2016,0,-6.0
235,Louisiana,2011,2016,0,-5.0
236,Louisiana,2012,2016,0,-4.0
237,Louisiana,2013,2016,0,-3.0
238,Louisiana,2014,2016,0,-2.0
239,Louisiana,2015,2016,0,-1.0
240,Louisiana,2016,2016,1,0.0
241,Louisiana,2017,2016,1,1.0
242,Louisiana,2018,2016,1,2.0
243,Louisiana,2019,2016,1,3.0



Texas (never expanded):


,state,year,expansion_year,post_expansion,event_time
559,Texas,2010,0,0,NaN
560,Texas,2011,0,0,NaN
561,Texas,2012,0,0,NaN
562,Texas,2013,0,0,NaN
563,Texas,2014,0,0,NaN
564,Texas,2015,0,0,NaN
565,Texas,2016,0,0,NaN
566,Texas,2017,0,0,NaN
567,Texas,2018,0,0,NaN
568,Texas,2019,0,0,NaN


---
## 3. Clean & Merge ACS Controls

In [5]:
# ── ACS State Controls via Census API ────────────────────────────────────────
#
# Get a FREE key in ~30 seconds at:  https://api.census.gov/data/key_signup.html
# Paste it below, then re-run this cell.
# ─────────────────────────────────────────────────────────────────────────────

CENSUS_API_KEY = 'PASTE_YOUR_KEY_HERE'   # <- replace with your key

acs_output_path = f'{RAW_DIR}/acs_state_controls.csv'

if os.path.exists(acs_output_path):
    df_acs = pd.read_csv(acs_output_path)
    df_acs['state_fips'] = df_acs['state_fips'].astype(str).str.zfill(2)
    print(f"Loaded cached ACS data: {df_acs.shape}  ({df_acs['year'].min()}-{df_acs['year'].max()})")

elif CENSUS_API_KEY and CENSUS_API_KEY != 'PASTE_YOUR_KEY_HERE':
    print("Fetching ACS data (takes ~2 min)...\n")
    df_acs = fetch_acs_data(api_key=CENSUS_API_KEY)
    if df_acs is not None:
        df_acs.to_csv(acs_output_path, index=False)
        print(f"\nSaved ACS data to {acs_output_path}")
    else:
        print("\n[WARN] All years failed -- check your API key.")
        df_acs = None
else:
    print("[WARN] No Census API key provided.")
    print("       Get one free at: https://api.census.gov/data/key_signup.html")
    print("       Analysis will proceed without demographic controls.")
    df_acs = None

Loaded cached ACS data: (676, 8)  (2010-2022)


## 4. Clean & Merge Mortality Data

CDC WONDER exports come in **two-part CSVs** because the classic UCD database
(1999–2020) and the expanded database (2018–present) are separate systems.

Files expected in `data/raw/`:

| Outcome | File (part 1) | File (part 2) |
|---------|--------------|---------------|
| All-cause | `cdc_wonder_allcause_mortality_2010_2020.csv` | `cdc_wonder_allcause_mortality_2018_2022.csv` |
| Diabetes | `cdc_wonder_diabetes_mortality_2010_2020.csv` | `cdc_wonder_diabetes_mortality_2018_2022.csv` |
| Maternal | `cdc_wonder_maternal_mortality_2010_2020.csv` | `cdc_wonder_maternal_mortality_2018_2022.csv` |

`parse_cdc_wonder_csv` handles the combining automatically:
uses part 1 for 2010–2017, part 2 for 2018–2022.

In [6]:
print("Loading and combining CDC WONDER mortality files...\n")

MORTALITY_CONFIGS = [
    {
        'file_old': f'{RAW_DIR}/cdc_wonder_allcause_mortality_2010_2020.csv',
        'file_new': f'{RAW_DIR}/cdc_wonder_allcause_mortality_2018_2022.csv',
        'prefix':   'allcause',
    },
    {
        'file_old': f'{RAW_DIR}/cdc_wonder_diabetes_mortality_2010_2020.csv',
        'file_new': f'{RAW_DIR}/cdc_wonder_diabetes_mortality_2018_2022.csv',
        'prefix':   'diabetes',
    },
    {
        'file_old': f'{RAW_DIR}/cdc_wonder_maternal_mortality_2010_2020.csv',
        'file_new': f'{RAW_DIR}/cdc_wonder_maternal_mortality_2018_2022.csv',
        'prefix':   'maternal',
    },
]

for cfg in MORTALITY_CONFIGS:
    try:
        df_mort = parse_cdc_wonder_csv(
            file_old=cfg['file_old'],
            file_new=cfg['file_new'],
            prefix=cfg['prefix'],
        )
        panel = panel.merge(df_mort, on=['state_fips', 'year'], how='left')
    except FileNotFoundError as e:
        print(f"  ⚠️  {e}")

print(f"\nPanel after mortality merge: {panel.shape}")
print(f"Outcome columns added: {[c for c in panel.columns if any(p in c for p in ['allcause','diabetes','maternal'])]}")

Loading and combining CDC WONDER mortality files...

  [OK] allcause     663 rows | 51 states | years 2010-2022 | 0% missing age-adj rate
  [OK] diabetes     663 rows | 51 states | years 2010-2022 | 0% missing age-adj rate
  [OK] maternal     386 rows | 38 states | years 2010-2022 | 23% missing age-adj rate

Panel after mortality merge: (663, 22)
Outcome columns added: ['allcause_deaths', 'allcause_population', 'allcause_crude_rate', 'allcause_age_adj_rate', 'diabetes_deaths', 'diabetes_population', 'diabetes_crude_rate', 'diabetes_age_adj_rate', 'maternal_deaths', 'maternal_population', 'maternal_crude_rate', 'maternal_age_adj_rate']


In [7]:
# ── Merge ACS controls if available ──────────────────────────────────────────
if df_acs is not None:
    acs_cols = ['state_fips', 'year', 'total_population', 'median_household_income',
                'poverty_rate', 'pct_white', 'pct_black', 'pct_hispanic']
    acs_merge = df_acs[[c for c in acs_cols if c in df_acs.columns]].copy()

    # ACS 1-year has no 2020 estimate (COVID) — interpolate within each state
    panel = panel.merge(acs_merge, on=['state_fips', 'year'], how='left')

    acs_numeric = [c for c in acs_merge.columns if c not in ['state_fips', 'year']]
    panel = panel.sort_values(['state_fips', 'year'])
    panel[acs_numeric] = panel.groupby('state_fips')[acs_numeric].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    panel = panel.reset_index(drop=True)
    print(f"✅ ACS controls merged.  Panel: {panel.shape}")
else:
    print("⚠️  ACS controls not available — proceeding without demographic controls.")
    print("   Re-run cell-8 after getting a Census API key.")

✅ ACS controls merged.  Panel: (663, 28)


---
## 5. Clean & Merge Diabetes Surveillance Data

In [8]:
# Diabetes Prevalence (BRFSS state-year time-series, 2011-2022)
print('Cleaning diabetes data...')
print()

prevalence_path = f'{RAW_DIR}/cdc_diabetes_prevalence.csv'

if os.path.exists(prevalence_path):
    df_prev = pd.read_csv(prevalence_path)
    df_prev['state_fips'] = df_prev['state_fips'].astype(str).str.zfill(2)
    df_prev['year'] = df_prev['year'].astype(int)
    panel = panel.merge(
        df_prev[['state_fips', 'year', 'diabetes_prevalence']],
        on=['state_fips', 'year'], how='left'
    )
    miss = panel['diabetes_prevalence'].isna().sum()
    ymin = df_prev['year'].min()
    ymax = df_prev['year'].max()
    n_states = df_prev['state_fips'].nunique()
    print(f'  [OK] diabetes prevalence  {len(df_prev):>4d} rows | {n_states} states | years {ymin}-{ymax}')
    print(f'       Panel missing after merge: {miss} (2010 has no BRFSS data)')
else:
    print(f'  Warning: Not found: {prevalence_path}')

# Incidence: single-year 2023 snapshot only - skip for panel DiD
print('  [SKIP] diabetes incidence - only 2023 snapshot available from CDC Atlas.')
print()
print(f'Panel after diabetes merge: {panel.shape}')


Cleaning diabetes data...

  [OK] diabetes prevalence   610 rows | 51 states | years 2011-2022
       Panel missing after merge: 53 (2010 has no BRFSS data)
  [SKIP] diabetes incidence - only 2023 snapshot available from CDC Atlas.

Panel after diabetes merge: (663, 29)


---
## 6. Clean & Merge Natality Data

In [9]:
natality_path = f'{RAW_DIR}/cdc_wonder_natality.txt'

if os.path.exists(natality_path):
    rows = []
    with open(natality_path, 'r') as f:
        header = [h.strip('"') for h in f.readline().strip().split('\t')]
        for line in f:
            stripped = line.strip()
            if stripped.startswith('---') or stripped.startswith('"---') or not stripped:
                break
            vals = [v.strip('"') for v in stripped.split('\t')]
            rows.append(vals)
    
    df_natality = pd.DataFrame(rows, columns=header)
    print(f"Natality raw: {df_natality.shape}")
    print(f"Columns: {df_natality.columns.tolist()}")
    
    # Rename key columns (adjust after seeing actual download)
    col_map = {}
    for col in df_natality.columns:
        col_lower = col.lower()
        if 'state code' in col_lower:
            col_map[col] = 'state_fips'
        elif col_lower == 'year' or 'year code' in col_lower:
            col_map[col] = 'year'
        elif 'births' in col_lower and 'birth weight' not in col_lower:
            col_map[col] = 'total_births'
        elif 'lbw' in col_lower or 'low birth weight' in col_lower:
            col_map[col] = 'low_birth_weight_pct'
        elif 'average birth weight' in col_lower:
            col_map[col] = 'avg_birth_weight'
    
    df_natality = df_natality.rename(columns=col_map)
    
    if 'state_fips' in df_natality.columns:
        df_natality['state_fips'] = df_natality['state_fips'].astype(str).str.zfill(2)
        df_natality['year'] = pd.to_numeric(df_natality['year'], errors='coerce')
        
        for col in ['total_births', 'low_birth_weight_pct', 'avg_birth_weight']:
            if col in df_natality.columns:
                df_natality[col] = pd.to_numeric(
                    df_natality[col].replace(['Suppressed', 'Unreliable', ''], np.nan),
                    errors='coerce'
                )
        
        natality_cols = ['state_fips', 'year'] + [
            c for c in ['total_births', 'low_birth_weight_pct', 'avg_birth_weight'] 
            if c in df_natality.columns
        ]
        df_natality_merge = df_natality[natality_cols].dropna(subset=['state_fips', 'year'])
        df_natality_merge['year'] = df_natality_merge['year'].astype(int)
        
        panel = panel.merge(df_natality_merge, on=['state_fips', 'year'], how='left')
        print(f"✅ Merged natality data. Panel: {panel.shape}")
    else:
        print("⚠️  Could not identify state_fips column. Check column names and adjust.")
else:
    print(f"⚠️  Natality file not found at {natality_path}")

⚠️  Natality file not found at ../data/raw/cdc_wonder_natality.txt


In [10]:
natality_path = f'{RAW_DIR}/cdc_wonder_natality.csv'

try:
    df_natality = parse_natality_csv(natality_path)
    panel = panel.merge(df_natality, on=['state_fips', 'year'], how='left')
    print(f"Panel after natality merge: {panel.shape}")
    print(f"Natality columns: {[c for c in df_natality.columns if c not in ['state_fips','year']]}")
except FileNotFoundError as e:
    print(f"⚠️  {e}")

  [OK] natality    663 rows | 51 states | years 2010-2022
Panel after natality merge: (663, 32)
Natality columns: ['total_births', 'birth_rate', 'avg_birth_weight_g']


In [11]:
brfss_path = f'{RAW_DIR}/brfss_health_access.csv'

if os.path.exists(brfss_path):
    df_brfss = pd.read_csv(brfss_path)
    print(f"BRFSS raw: {df_brfss.shape}")
    print(f"Columns: {df_brfss.columns.tolist()[:15]}")
    
    # BRFSS format varies depending on download method
    # Example for CDC SODAPI format:
    if 'locationabbr' in df_brfss.columns:
        try:
            import us
            state_abbrevs = [s.abbr for s in us.states.STATES_AND_DC]
            df_brfss = df_brfss[df_brfss['locationabbr'].isin(state_abbrevs)].copy()
            abbr_to_fips = {s.abbr: s.fips for s in us.states.STATES_AND_DC}
            df_brfss['state_fips'] = df_brfss['locationabbr'].map(abbr_to_fips)
        except ImportError:
            print("Install 'us' package: pip install us")
    
    print("\n⚠️  BRFSS data format varies. Review columns above and adjust as needed.")
    display(df_brfss.head())
else:
    print(f"⚠️  BRFSS file not found. This is optional — proceeding without it.")

⚠️  BRFSS file not found. This is optional — proceeding without it.


---
## 8. Final Panel Validation

In [12]:
print("=" * 60)
print("FINAL PANEL SUMMARY")
print("=" * 60)

print(f"\nShape: {panel.shape}")
print(f"States: {panel['state'].nunique()}")
print(f"Years: {panel['year'].min()}–{panel['year'].max()}")
print(f"Obs per state: {panel.groupby('state').size().unique()}")

print(f"\nColumns ({len(panel.columns)}):")
for col in panel.columns:
    non_null = panel[col].notna().sum()
    pct = non_null / len(panel) * 100
    print(f"  {col:40s} {non_null:>5d} non-null ({pct:5.1f}%)")

FINAL PANEL SUMMARY

Shape: (663, 32)
States: 51
Years: 2010–2022
Obs per state: [13]

Columns (32):
  state                                      663 non-null (100.0%)
  state_fips                                 663 non-null (100.0%)
  year                                       663 non-null (100.0%)
  expansion_year                             663 non-null (100.0%)
  ever_expanded                              663 non-null (100.0%)
  cohort                                     663 non-null (100.0%)
  post_expansion                             663 non-null (100.0%)
  event_time                                 533 non-null ( 80.4%)
  cohort_group                               663 non-null (100.0%)
  treat_x_post                               663 non-null (100.0%)
  allcause_deaths                            663 non-null (100.0%)
  allcause_population                        663 non-null (100.0%)


  allcause_crude_rate                        663 non-null (100.0%)
  allcause_age_adj_rate                      663 non-null (100.0%)
  diabetes_deaths                            663 non-null (100.0%)
  diabetes_population                        663 non-null (100.0%)
  diabetes_crude_rate                        663 non-null (100.0%)
  diabetes_age_adj_rate                      663 non-null (100.0%)
  maternal_deaths                            386 non-null ( 58.2%)
  maternal_population                        386 non-null ( 58.2%)
  maternal_crude_rate                        299 non-null ( 45.1%)
  maternal_age_adj_rate                      299 non-null ( 45.1%)
  total_population                           663 non-null (100.0%)
  median_household_income                    663 non-null (100.0%)
  poverty_rate                               663 non-null (100.0%)
  pct_white                                  663 non-null (100.0%)
  pct_black                                  663 non-null (100

In [13]:
# Check balance
expected_obs = panel['state'].nunique() * len(YEARS)
actual_obs = len(panel)

if actual_obs == expected_obs:
    print(f"✅ Panel is balanced: {actual_obs} observations (expected {expected_obs})")
else:
    print(f"⚠️  Panel is UNBALANCED: {actual_obs} observations (expected {expected_obs})")
    state_year_counts = panel.groupby('state').size()
    problem_states = state_year_counts[state_year_counts != len(YEARS)]
    if len(problem_states) > 0:
        print(f"States with missing years: {problem_states.to_dict()}")

✅ Panel is balanced: 663 observations (expected 663)


In [14]:
# Treatment balance check
print("Treatment status by year:\n")
treat_by_year = panel.groupby('year').agg(
    n_treated=('post_expansion', 'sum'),
    n_untreated=('post_expansion', lambda x: (x == 0).sum()),
    n_never_expanded=('ever_expanded', lambda x: (x == 0).sum())
).reset_index()

print(treat_by_year.to_string(index=False))

Treatment status by year:

 year  n_treated  n_untreated  n_never_expanded
 2010          0           51                10
 2011          0           51                10
 2012          0           51                10
 2013          0           51                10
 2014         27           24                10
 2015         30           21                10
 2016         32           19                10
 2017         32           19                10
 2018         32           19                10
 2019         34           17                10
 2020         37           14                10
 2021         39           12                10
 2022         39           12                10


In [15]:
# Descriptive statistics by expansion status
print("\nDescriptive Statistics: Expansion vs Non-Expansion States (Pre-period, 2010-2013)\n")

pre_period = panel[panel['year'].between(2010, 2013)].copy()

# Identify numeric columns (excluding identifiers and treatment vars)
exclude_cols = ['state', 'state_fips', 'year', 'expansion_year', 'ever_expanded', 
                'cohort', 'post_expansion', 'event_time', 'cohort_group', 'treat_x_post']
numeric_cols = [c for c in pre_period.select_dtypes(include=[np.number]).columns 
                if c not in exclude_cols]

if numeric_cols:
    comparison = pre_period.groupby('ever_expanded')[numeric_cols].mean().T
    comparison.columns = ['Non-Expansion', 'Expansion']
    comparison['Difference'] = comparison['Expansion'] - comparison['Non-Expansion']
    display(comparison.round(2))
else:
    print("No outcome data available yet. Download CDC data and re-run.")


Descriptive Statistics: Expansion vs Non-Expansion States (Pre-period, 2010-2013)



,Non-Expansion,Expansion,Difference
allcause_deaths,68223.15,45092.92,-23130.23
allcause_population,8294995.68,5601100.65,-2693895.02
allcause_crude_rate,870.14,832.27,-37.87
allcause_age_adj_rate,809.76,743.90,-65.86
diabetes_deaths,1953.08,1306.64,-646.43
diabetes_population,8294995.68,5601100.65,-2693895.02
diabetes_crude_rate,24.32,24.06,-0.25
diabetes_age_adj_rate,22.34,21.42,-0.92
maternal_deaths,47.11,28.21,-18.91
maternal_population,5382227.37,4771128.86,-611098.51


---
## 9. Save Final Panel

In [16]:
# Sort and save
panel = panel.sort_values(['state', 'year']).reset_index(drop=True)

output_path = f'{PROCESSED_DIR}/analysis_panel.csv'
panel.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / 1024
print(f"✅ Saved final panel to {output_path}")
print(f"   Size: {file_size:.1f} KB")
print(f"   Shape: {panel.shape}")
print(f"   Columns: {list(panel.columns)}")

✅ Saved final panel to ../data/processed/analysis_panel.csv
   Size: 114.2 KB
   Shape: (663, 32)
   Columns: ['state', 'state_fips', 'year', 'expansion_year', 'ever_expanded', 'cohort', 'post_expansion', 'event_time', 'cohort_group', 'treat_x_post', 'allcause_deaths', 'allcause_population', 'allcause_crude_rate', 'allcause_age_adj_rate', 'diabetes_deaths', 'diabetes_population', 'diabetes_crude_rate', 'diabetes_age_adj_rate', 'maternal_deaths', 'maternal_population', 'maternal_crude_rate', 'maternal_age_adj_rate', 'total_population', 'median_household_income', 'poverty_rate', 'pct_white', 'pct_black', 'pct_hispanic', 'diabetes_prevalence', 'total_births', 'birth_rate', 'avg_birth_weight_g']


In [17]:
# Final peek
print("Sample — Ohio (expanded 2014):\n")
display(panel[panel['state'] == 'Ohio'].head())

print("\nSample — Missouri (expanded 2021):\n")
display(panel[panel['state'] == 'Missouri'].head())

print("\nSample — Florida (never expanded):\n")
display(panel[panel['state'] == 'Florida'].head())

Sample — Ohio (expanded 2014):



,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post,allcause_deaths,allcause_population,allcause_crude_rate,allcause_age_adj_rate,diabetes_deaths,...,diabetes_age_adj_rate,maternal_deaths,maternal_population,maternal_crude_rate,maternal_age_adj_rate,total_population,median_household_income,poverty_rate,pct_white,pct_black,pct_hispanic,diabetes_prevalence,total_births,birth_rate,avg_birth_weight_g
455,Ohio,39,2010,2014,1,Early (2014),0,-4.0,2014,0,108711,11536504,942.3,815.7,3470,...,25.8,40.0,5904348.0,0.7,0.7,11536182.0,45090.0,15.85,81.05,12.09,3.08,NaN,139128,12.06,3257.94
456,Ohio,39,2011,2014,1,Early (2014),0,-3.0,2014,0,111427,11544951,965.2,821.8,3668,...,26.8,35.0,5905599.0,0.6,0.7,11544951.0,45749.0,16.43,80.92,11.96,3.15,10.0,137918,11.95,3259.45
457,Ohio,39,2012,2014,1,Early (2014),0,-2.0,2014,0,112498,11544225,974.5,817.9,3618,...,26.1,39.0,5901605.0,0.7,0.7,11544225.0,46829.0,16.25,80.64,12.06,3.24,11.7,138483,12.00,3261.67
458,Ohio,39,2013,2014,1,Early (2014),0,-1.0,2014,0,113258,11570808,978.8,811.2,3563,...,25.4,30.0,5911149.0,0.5,0.6,11570808.0,48081.0,15.97,80.34,11.99,3.31,10.4,138936,12.01,3266.24
459,Ohio,39,2014,2014,1,Early (2014),1,0.0,2014,1,114509,11594163,987.6,810.0,3641,...,25.7,22.0,5919391.0,0.4,0.4,11594163.0,49308.0,15.84,80.02,12.09,3.44,11.7,139467,12.03,3266.98



Sample — Missouri (expanded 2021):



,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post,allcause_deaths,allcause_population,allcause_crude_rate,allcause_age_adj_rate,diabetes_deaths,...,diabetes_age_adj_rate,maternal_deaths,maternal_population,maternal_crude_rate,maternal_age_adj_rate,total_population,median_household_income,poverty_rate,pct_white,pct_black,pct_hispanic,diabetes_prevalence,total_births,birth_rate,avg_birth_weight_g
325,Missouri,29,2010,2021,1,Late (2021),0,-11.0,2021,0,55281,5988927,923.1,819.5,1425,...,21.2,12.0,3055450.0,NaN,NaN,5996231.0,44301.0,15.27,80.96,11.61,3.56,NaN,76759,12.82,3270.81
326,Missouri,29,2011,2021,1,Late (2021),0,-10.0,2021,0,55848,6010688,929.1,812.0,1438,...,20.8,17.0,3066254.0,NaN,NaN,6010688.0,45247.0,15.78,80.79,11.38,3.60,10.2,76117,12.66,3280.25
327,Missouri,29,2012,2021,1,Late (2021),0,-9.0,2021,0,56094,6021988,931.5,803.0,1377,...,19.6,34.0,3070952.0,1.1,1.2,6021988.0,45321.0,16.23,80.52,11.39,3.68,10.7,75446,12.53,3282.88
328,Missouri,29,2013,2021,1,Late (2021),0,-8.0,2021,0,57444,6044171,950.4,807.7,1477,...,20.5,30.0,3080214.0,1.0,1.1,6044171.0,46931.0,15.89,80.39,11.36,3.80,9.6,75296,12.46,3284.08
329,Missouri,29,2014,2021,1,Late (2021),0,-7.0,2021,0,58320,6063589,961.8,807.0,1423,...,19.4,34.0,3089367.0,1.1,1.2,6063589.0,48363.0,15.46,79.95,11.61,3.85,11.1,75360,12.43,3289.49



Sample — Florida (never expanded):



,state,state_fips,year,expansion_year,ever_expanded,cohort,post_expansion,event_time,cohort_group,treat_x_post,allcause_deaths,allcause_population,allcause_crude_rate,allcause_age_adj_rate,diabetes_deaths,...,diabetes_age_adj_rate,maternal_deaths,maternal_population,maternal_crude_rate,maternal_age_adj_rate,total_population,median_household_income,poverty_rate,pct_white,pct_black,pct_hispanic,diabetes_prevalence,total_births,birth_rate,avg_birth_weight_g
117,Florida,12,2010,0,0,Never,0,NaN,0,0,173791,18801310,924.4,701.1,5024,...,20.1,47.0,9611955.0,0.5,0.6,18843326.0,44409.0,16.53,57.76,15.26,22.57,NaN,214590,11.41,3229.88
118,Florida,12,2011,0,0,Never,0,NaN,0,0,173976,19057542,912.9,677.1,5093,...,19.7,50.0,9736166.0,0.5,0.6,19057542.0,44299.0,17.01,57.34,15.34,22.85,10.4,213414,11.20,3233.13
119,Florida,12,2012,0,0,Never,0,NaN,0,0,177291,19317568,917.8,669.9,5092,...,19.2,64.0,9869741.0,0.6,0.8,19317568.0,45040.0,17.12,56.77,15.32,23.21,11.4,213148,11.03,3241.15
120,Florida,12,2013,0,0,Never,0,NaN,0,0,181112,19552860,926.3,663.4,5238,...,19.2,83.0,9990213.0,0.8,1.0,19552860.0,46036.0,17.01,56.22,15.45,23.62,11.2,215407,11.02,3243.87
121,Florida,12,2014,0,0,Never,0,NaN,0,0,185956,19893297,934.8,662.0,5371,...,19.2,62.0,10170011.0,0.6,0.7,19893297.0,47463.0,16.50,55.58,15.50,24.07,11.2,219991,11.06,3243.77


---
## Next Steps

The panel is ready. Proceed to:
- **`03_eda.ipynb`** — Exploratory data analysis, descriptive statistics, pre-trends visualization
- **`04_did_analysis.ipynb`** — Difference-in-differences estimation